In [ ]:
import os
from google.oauth2.service_account import Credentials
from googleapiclient import discovery

# ============================================================
# BURAYA KENDİ CREDENTIALS PATH'İNİ YAZ
CREDENTIALS_PATH = "/content/google_drive_credentials.json"  # <-- değiştir
# ============================================================

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
credentials = Credentials.from_service_account_file(CREDENTIALS_PATH, scopes=SCOPES)
service = discovery.build('drive', 'v3', credentials=credentials)

def list_files(query, page_size=100):
    all_files = []
    page_token = None
    while True:
        params = dict(
            q=query,
            spaces='drive',
            fields="nextPageToken, files(id, name, mimeType, parents)",
            pageSize=page_size,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
        )
        if page_token:
            params['pageToken'] = page_token
        results = service.files().list(**params).execute()
        all_files.extend(results.get('files', []))
        page_token = results.get('nextPageToken')
        if not page_token:
            break
    return all_files

def explore(folder_id, folder_name, depth=0, max_depth=6, visited=None):
    if visited is None:
        visited = set()
    if folder_id in visited or depth > max_depth:
        return
    visited.add(folder_id)

    indent = "  " * depth
    print(f"{indent}📁 {folder_name}/")

    children = list_files(f"'{folder_id}' in parents and trashed=false", page_size=200)

    folders = [f for f in children if f['mimeType'] == 'application/vnd.google-apps.folder']
    files   = [f for f in children if f['mimeType'] != 'application/vnd.google-apps.folder']

    for f in sorted(folders, key=lambda x: x['name']):
        explore(f['id'], f['name'], depth + 1, max_depth, visited)

    # Sadece .pth dosyalarını ve video dosyalarını göster
    for f in sorted(files, key=lambda x: x['name']):
        name = f['name']
        if any(name.endswith(ext) for ext in ['.pth', '.pt', '.h5', '.pkl', '.mp4', '.avi', '.mkv']):
            print(f"{indent}  📄 {name}")

# ============================================================
# sharedWithMe klasörler
print("=" * 60)
print("SHARED WITH ME KLASÖRLER")
print("=" * 60)
shared_folders = list_files("sharedWithMe=true and mimeType='application/vnd.google-apps.folder' and trashed=false")
for f in shared_folders:
    print(f"\n→ ROOT KLASÖR: {f['name']}  (id: {f['id']})")
    explore(f['id'], f['name'], depth=1)

# Root klasörler
print("\n" + "=" * 60)
print("MY DRIVE KÖK KLASÖRLER")
print("=" * 60)
root_folders = list_files("'root' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false")
for f in root_folders:
    print(f"\n→ ROOT KLASÖR: {f['name']}  (id: {f['id']})")
    explore(f['id'], f['name'], depth=1)

In [ ]:
import torch

# 1. CNN Model dosyasının yolu
CNN_PATH = "/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors-CNN/training_final/cnn_multilabel_final_model/best_model.pth"

print("  [INFO] .pth dosyası okunuyor...")
checkpoint = torch.load(CNN_PATH, map_location=torch.device('cpu'))

# 2. Checkpoint yapısını inceleme
if isinstance(checkpoint, dict):
    print("\n==================================================")
    print(f"Sözlük Anahtarları (Keys): {list(checkpoint.keys())}")
    print("==================================================")

    # Ağırlıkların tutulduğu anahtarı bul (Önceki çıktılarına göre 'model_state')
    state_dict_key = 'model_state' if 'model_state' in checkpoint else ('state_dict' if 'state_dict' in checkpoint else None)
    weights_dict = checkpoint[state_dict_key] if state_dict_key else checkpoint

    print(f"\n--- CNN Katmanları ve Tensor Boyutları ({state_dict_key if state_dict_key else 'Doğrudan Ağırlıklar'}) ---")

    layer_names = list(weights_dict.keys())

    # Tüm katmanları listele
    for layer_name, weights in weights_dict.items():
        if hasattr(weights, 'shape'):
            print(f"Katman: {layer_name:<40} | Boyut (Shape): {str(list(weights.shape)):<25}")

    print("\n==================================================")
    print("               MODEL PARAMETRE ANALİZİ            ")
    print("==================================================")

    # Girdi analizini yap (Genellikle ilk conv katmanıdır)
    first_layer = layer_names[0]
    first_shape = list(weights_dict[first_layer].shape)
    print(f"➔ İlk Katman: {first_layer}")
    print(f"➔ İlk Katman Boyutu: {first_shape}")

    if len(first_shape) == 5: # 3D CNN yapısı: [Out_Channels, In_Channels, Kernel_T, Kernel_H, Kernel_W]
        print(f"  └─ Girdi Tipi: 3D Video Tensor")
        print(f"  └─ Beklenen Girdi Kanalı (RGB): {first_shape[1]}")
    elif len(first_shape) == 4: # 2D CNN yapısı: [Out_Channels, In_Channels, Kernel_H, Kernel_W]
        print(f"  └─ Girdi Tipi: 2D Görsel Tensor")
        print(f"  └─ Beklenen Girdi Kanalı: {first_shape[1]}")

    # Çıktı analizini yap (Son linear/fc katmanının weight tensoru)
    # Listeyi tersten tarayarak en son 'weight' barındıran fc/classifier katmanını bulalım
    fc_layer = None
    for name in reversed(layer_names):
        if 'fc.weight' in name or 'classifier.weight' in name or 'linear.weight' in name or (name.endswith('.weight') and 'fc' in name):
            fc_layer = name
            break

    if fc_layer:
        fc_shape = list(weights_dict[fc_layer].shape)
        print(f"\n➔ Son Katman (Sınıflandırıcı): {fc_layer}")
        print(f"➔ Son Katman Boyutu: {fc_shape}")
        print(f"  └─ Çıkış Sınıf Sayısı (Num Classes): {fc_shape[0]}")
    else:
        # Alternatif olarak son ağırlığın ilk boyutuna bak
        last_layer = [n for n in layer_names if weights_dict[n].ndim > 1][-1]
        print(f"\n➔ Tespit Edilen Son Ağırlık Katmanı: {last_layer}")
        print(f"  └─ Tahmini Sınıf Sayısı: {weights_dict[last_layer].shape[0]}")

else:
    print("\n[INFO] Dosya bir sözlük değil, doğrudan derlenmiş PyTorch nesnesi.")
    print("==================================================")
    print(checkpoint)
    if hasattr(checkpoint, 'eval'):
        print("\n--- Model Mimarisi ---")
        print(checkpoint)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BIM437 · AUTO-LABEL & SAMPLE PIPELINE
# LTD'deki tüm 2dk videoları CNN ile sınıflandırır
# Her sınıf × mevsim için 40 video seçer (toplam 320)
# Her video için 4 görsel (30sn aralıklarla) + video adı kaydeder
# L4 GPU optimize · Batch inference · Checkpoint resume
# ═══════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision.models.video import r3d_18
from torch.amp import autocast
from pathlib import Path
from collections import defaultdict
import json, time, random, re
from datetime import datetime

# ──────────────────────────────────────────────────────────
# AYARLAR
# ──────────────────────────────────────────────────────────
LTD_DIR    = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
CNN_PATH   = "/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors-CNN/training_final/cnn_multilabel_final_model/best_model.pth"
CRAE_PATH  = "/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors-CRAE/models/crae_winter_finetuned2.pth"

OUTPUT_DIR = Path("/content/drive/MyDrive/archive/auto_labeled_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = OUTPUT_DIR / "label_cache.json"   # checkpoint
RESULT_JSON = OUTPUT_DIR / "final_selection.json"

CLASSES        = ["normal","trespassing","loitering","object_abandonment"]
DEVICE         = torch.device("cuda")
N_FRAMES       = 16
CNN_SIZE       = 112
CRAE_SIZE      = 128
CRAE_THRESHOLD = 0.015
CONF_THRESHOLD = 0.5       # CNN multi-label eşiği
BATCH_SIZE     = 8         # L4 için güvenli; 16'ya kadar çıkabilir

# Mevsim ayları (kuzey yarımküre)
SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}

TARGET_PER_BUCKET = 40     # her sınıf×mevsim için
SAMPLES_PER_VIDEO = 4      # 30sn aralıklarla 4 frame görseli
VIDEO_LEN_SEC     = 120    # 2 dakika varsayımı

CNN_MEAN  = np.array([0.485,0.456,0.406], dtype=np.float32)
CNN_STD   = np.array([0.229,0.224,0.225], dtype=np.float32)

COLORS = {
    "normal":             "#4CAF50",
    "trespassing":        "#F44336",
    "loitering":          "#FF9800",
    "object_abandonment": "#9C27B0",
}
BG, PANEL, BORDER, TEXT, MUTED = "#0D1117","#161B22","#21262D","#C9D1D9","#8B949E"

# ──────────────────────────────────────────────────────────
# MODEL YÜKLEME
# ──────────────────────────────────────────────────────────
print("═"*60)
print("  BIM437  ·  AUTO-LABEL & SAMPLE PIPELINE")
print("═"*60)

cnn = r3d_18(weights=None)
cnn.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(cnn.fc.in_features, 4))
ckpt = torch.load(CNN_PATH, map_location=DEVICE)
cnn.load_state_dict(ckpt["model_state"])
cnn = cnn.to(DEVICE).eval()
print(f"✓ CNN yüklendi  (val_f1={ckpt['val_f1']:.4f})")
print(f"✓ Device: {DEVICE}  · Batch: {BATCH_SIZE}\n")

# ──────────────────────────────────────────────────────────
# YARDIMCI FONKSİYONLAR
# ──────────────────────────────────────────────────────────
def parse_season(day_folder_name: str):
    """20YYMMDD veya YYYYMMDD klasör adından mevsim çıkar."""
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", day_folder_name)
    if not m: return None, None
    year, month = int(m.group(1)), int(m.group(2))
    if month in SUMMER_MONTHS: return "summer", year
    if month in WINTER_MONTHS: return "winter", year
    return None, year   # ara mevsim — atla

def extract_frames_at(cap, start_frame, n=N_FRAMES):
    """Belirli pozisyondan n frame oku."""
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames = []
    for _ in range(n):
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    return frames if len(frames) == n else None

def preprocess_cnn_batch(frame_lists):
    """Birden fazla 16-frame klibini tek tensor'a (B,3,16,112,112) çevir."""
    out = []
    for frames in frame_lists:
        fm = [cv2.resize(f, (CNN_SIZE, CNN_SIZE)) for f in frames]
        arr = (np.stack(fm).astype(np.float32)/255.0 - CNN_MEAN) / CNN_STD
        out.append(arr.transpose(3,0,1,2))   # (3,16,H,W)
    return torch.from_numpy(np.stack(out)).float()

@torch.no_grad()
def infer_cnn_batch(tensor_batch):
    """Batch inference → (B,4) probs."""
    tensor_batch = tensor_batch.to(DEVICE, non_blocking=True)
    with autocast('cuda'):
        logits = cnn(tensor_batch)
    return torch.sigmoid(logits).cpu().numpy()

# ──────────────────────────────────────────────────────────
# VIDEO ETİKETLEME
# ──────────────────────────────────────────────────────────
def label_video(mp4_path, n_samples=3):
    """
    Bir 2dk videodan n_samples noktada 16-frame klip al, CNN'den geçir.
    Her sınıf için max prob'u döndür → video düzeyi etiket.
    """
    cap = cv2.VideoCapture(str(mp4_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
    if total < N_FRAMES + 1:
        cap.release(); return None, None, None

    # Eşit aralıklı örnekleme noktaları
    sample_starts = np.linspace(0, max(0, total-N_FRAMES-1),
                                 n_samples, dtype=int)

    clips = []
    for s in sample_starts:
        fr = extract_frames_at(cap, int(s))
        if fr: clips.append(fr)
    cap.release()

    if not clips: return None, None, None

    # Batch inference
    tens = preprocess_cnn_batch(clips)
    probs_arr = infer_cnn_batch(tens)        # (n_samples, 4)
    max_probs = probs_arr.max(axis=0)        # video düzeyi: max prob per class

    # Dominant sınıf: en yüksek prob (multi-label olsa da en kuvvetlisi)
    dominant = CLASSES[int(max_probs.argmax())]
    confidence = float(max_probs.max())
    return dominant, confidence, max_probs.tolist()

# ──────────────────────────────────────────────────────────
# AŞAMA 1: TÜM VİDEOLARI TARA VE ETİKETLE
# ──────────────────────────────────────────────────────────
def scan_and_label_all():
    """Tüm yaz/kış videoları sınıflandır. Checkpoint'li."""
    # Cache yükle
    cache = {}
    if CACHE_FILE.exists():
        with open(CACHE_FILE) as f: cache = json.load(f)
        print(f"  📂 Cache yüklendi: {len(cache)} video zaten etiketli")

    # Tüm gün klasörlerini tara
    day_dirs = sorted([d for d in LTD_DIR.iterdir() if d.is_dir()])
    print(f"  📁 Toplam {len(day_dirs)} gün klasörü\n")

    # Sadece yaz/kış olan günleri filtrele
    valid_days = []
    for d in day_dirs:
        season, year = parse_season(d.name)
        if season in ("summer","winter") and year in (2020, 2021):
            valid_days.append((d, season, year))

    print(f"  ✓ Yaz/Kış 2020-2021 günler: {len(valid_days)}\n")

    # Tüm mp4'leri topla
    all_videos = []
    for d, season, year in valid_days:
        for mp4 in d.glob("*.mp4"):
            all_videos.append((mp4, season, year, d.name))

    print(f"  🎞  Toplam video: {len(all_videos)}")
    print(f"  ⏭  Atlanacak (cache): {sum(1 for v in all_videos if str(v[0]) in cache)}")
    print(f"  🔄 Etiketlenecek: {sum(1 for v in all_videos if str(v[0]) not in cache)}\n")

    # Etiketle
    t0 = time.time()
    processed = 0
    for idx, (mp4, season, year, day_name) in enumerate(all_videos):
        key = str(mp4)
        if key in cache:
            continue

        try:
            label, conf, probs = label_video(mp4)
            if label is None:
                cache[key] = {"error": "no_frames"}
            else:
                cache[key] = {
                    "label": label,
                    "confidence": conf,
                    "probs": probs,
                    "season": season,
                    "year": year,
                    "day": day_name,
                    "name": mp4.name,
                }
            processed += 1

            # Her 20 videoda checkpoint kaydet
            if processed % 20 == 0:
                with open(CACHE_FILE, "w") as f: json.dump(cache, f)
                elapsed = time.time() - t0
                rate = processed / elapsed
                remaining = sum(1 for v in all_videos if str(v[0]) not in cache)
                eta = remaining / rate if rate > 0 else 0
                print(f"  [{idx+1:>4}/{len(all_videos)}] "
                      f"{mp4.name[:35]:<35} → {label or 'ERR':<20} "
                      f"({conf or 0:.2f})  "
                      f"| {rate:.1f} vid/sn · ETA {eta/60:.1f}dk")
        except Exception as e:
            cache[key] = {"error": str(e)[:100]}
            print(f"  ✗ {mp4.name}: {e}")

    # Son kaydet
    with open(CACHE_FILE, "w") as f: json.dump(cache, f)
    print(f"\n  ✓ Etiketleme tamam · {processed} yeni · {len(cache)} toplam")
    print(f"  ⏱  Süre: {(time.time()-t0)/60:.1f} dakika\n")
    return cache

# ──────────────────────────────────────────────────────────
# AŞAMA 2: BUCKET'LARA AYIRMA
# ──────────────────────────────────────────────────────────
def select_buckets(cache):
    """Her sınıf×mevsim için en yüksek güvenlikli 40 video seç."""
    buckets = defaultdict(list)
    for path, info in cache.items():
        if "error" in info: continue
        if info.get("season") not in ("summer","winter"): continue
        key = (info["label"], info["season"])
        buckets[key].append({"path": path, **info})

    # Güvenlik skoruna göre sırala ve top-40 al
    selection = {}
    for (cls, season), vids in buckets.items():
        vids.sort(key=lambda x: -x["confidence"])
        selection[f"{cls}_{season}"] = vids[:TARGET_PER_BUCKET]

    print("  📊 Bucket Durumu:")
    print("  " + "─"*55)
    for cls in CLASSES:
        for season in ("summer","winter"):
            k = f"{cls}_{season}"
            n = len(selection.get(k, []))
            warn = "" if n >= TARGET_PER_BUCKET else f"  ⚠ ({TARGET_PER_BUCKET-n} eksik)"
            print(f"  {cls:<22} · {season:<6} : {n:>3} video{warn}")
    print()
    return selection

# ──────────────────────────────────────────────────────────
# AŞAMA 3: HER VIDEO İÇİN 4 GÖRSEL ÇIKAR
# ──────────────────────────────────────────────────────────
def render_video_samples(video_info, out_dir):
    """Bir videodan 30sn aralıklarla 4 frame çıkar, tek panelde kaydet."""
    mp4 = Path(video_info["path"])
    cap = cv2.VideoCapture(str(mp4))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # 30sn aralıklı 4 nokta: 0, 30, 60, 90 sn (varsa)
    sample_secs = [0, 30, 60, 90]
    sample_frames = []
    for sec in sample_secs:
        fr_idx = min(int(sec * fps), total-1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, fr_idx)
        ret, f = cap.read()
        if ret:
            sample_frames.append((sec, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    cap.release()
    if not sample_frames: return None

    # Panel oluştur: 1×4 grid
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), facecolor=BG)
    color = COLORS[video_info["label"]]

    for ax, (sec, frame) in zip(axes, sample_frames):
        ax.imshow(frame)
        ax.set_facecolor(PANEL)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"t = {sec}s", fontsize=9, color=MUTED,
                     fontfamily='monospace', pad=4)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER); sp.set_linewidth(0.6)

    # Üst başlık: video adı + güven
    title = (f"{video_info['name']}   ·   {video_info['day']}   ·   "
             f"conf={video_info['confidence']:.3f}")
    fig.suptitle(title, fontsize=10, color=TEXT,
                 fontfamily='monospace', y=0.995)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out_path = out_dir / f"{mp4.stem}.png"
    plt.savefig(str(out_path), dpi=110, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    return str(out_path)

def export_all_samples(selection):
    """Tüm seçili videoları sınıf/mevsim klasörlerine kaydet."""
    summary = {}
    t0 = time.time()
    total_videos = sum(len(v) for v in selection.values())
    done = 0

    print(f"  🖼  {total_videos} video için görsel üretiliyor...\n")

    for bucket_name, videos in selection.items():
        bucket_dir = OUTPUT_DIR / bucket_name
        bucket_dir.mkdir(exist_ok=True)
        summary[bucket_name] = []

        # Bucket içinde video listesi metin dosyası
        list_file = bucket_dir / "_video_list.txt"
        with open(list_file, "w", encoding='utf-8') as lf:
            cls, season = bucket_name.rsplit("_", 1)
            lf.write(f"# {cls.upper()} · {season.upper()}\n")
            lf.write(f"# Toplam: {len(videos)} video\n")
            lf.write(f"# Oluşturma: {datetime.now().isoformat()}\n\n")

            for i, v in enumerate(videos, 1):
                img_path = render_video_samples(v, bucket_dir)
                lf.write(f"{i:>3}. {v['name']}\n")
                lf.write(f"     gün: {v['day']}  ·  conf: {v['confidence']:.3f}\n")
                lf.write(f"     yol: {v['path']}\n\n")

                summary[bucket_name].append({
                    "name": v["name"],
                    "path": v["path"],
                    "day": v["day"],
                    "confidence": v["confidence"],
                    "probs": v["probs"],
                    "image": img_path,
                })
                done += 1
                if done % 10 == 0:
                    rate = done / (time.time()-t0)
                    eta  = (total_videos-done) / rate if rate>0 else 0
                    print(f"  [{done:>3}/{total_videos}]  {bucket_name:<32}  "
                          f"{rate:.1f} vid/sn · ETA {eta:.0f}sn")

    # Final JSON
    with open(RESULT_JSON, "w", encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"\n  ✓ Tüm görseller kaydedildi: {OUTPUT_DIR}")
    print(f"  📄 Özet JSON: {RESULT_JSON}\n")
    return summary

# ──────────────────────────────────────────────────────────
# AŞAMA 4: ANA RAPOR — Kategoriler altında video listeleri
# ──────────────────────────────────────────────────────────
def print_final_report(summary):
    print("═"*60)
    print("  FİNAL SEÇİM RAPORU")
    print("═"*60 + "\n")

    for cls in CLASSES:
        for season in ("summer","winter"):
            k = f"{cls}_{season}"
            vids = summary.get(k, [])
            color_label = f"{cls.upper()} · {season.upper()}"
            print(f"┌─ {color_label}  ({len(vids)} video) " + "─"*(40-len(color_label)))
            for i, v in enumerate(vids, 1):
                print(f"│ {i:>3}. {v['name']:<30}  "
                      f"conf={v['confidence']:.3f}  ({v['day']})")
            print("└" + "─"*58 + "\n")

# ──────────────────────────────────────────────────────────
# ÇALIŞTIR
# ──────────────────────────────────────────────────────────
print("\n[1/3] Tüm videoları etiketleme")
print("─"*60)
cache = scan_and_label_all()

print("[2/3] Bucket'lara ayırma (her sınıf×mevsim için top-40)")
print("─"*60)
selection = select_buckets(cache)

print("[3/3] Görsel + video listesi üretme")
print("─"*60)
summary = export_all_samples(selection)

print_final_report(summary)

print("═"*60)
print(f"  TAMAM  ·  Çıktı: {OUTPUT_DIR}")
print("═"*60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BIM437 · AUTO-LABEL & SAMPLE PIPELINE  (v2 — erken durdurma)
# Her bucket 40'a ulaşınca tarama durur
# Gün klasörleri shuffle edilir → yaz/kış paralel dolar
# ═══════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, numpy as np, cv2
import matplotlib.pyplot as plt
from torchvision.models.video import r3d_18
from torch.amp import autocast
from pathlib import Path
from collections import defaultdict
import json, time, random, re
from datetime import datetime

# ──────────────────────────────────────────────────────────
# AYARLAR
# ──────────────────────────────────────────────────────────
LTD_DIR    = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
CNN_PATH   = "/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors-CNN/training_final/cnn_multilabel_final_model/best_model.pth"

OUTPUT_DIR = Path("/content/drive/MyDrive/archive/auto_labeled_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE  = OUTPUT_DIR / "label_cache.json"
RESULT_JSON = OUTPUT_DIR / "final_selection.json"

CLASSES        = ["normal","trespassing","loitering","object_abandonment"]
DEVICE         = torch.device("cuda")
N_FRAMES       = 16
CNN_SIZE       = 112
BATCH_SIZE     = 8

SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}

TARGET_PER_BUCKET = 40
SAMPLES_PER_VIDEO = 4
MIN_CONFIDENCE    = 0.5     # Bu eşiğin altındaki videolar bucket'a alınmaz
SHUFFLE_SEED      = 42

CNN_MEAN  = np.array([0.485,0.456,0.406], dtype=np.float32)
CNN_STD   = np.array([0.229,0.224,0.225], dtype=np.float32)

COLORS = {"normal":"#4CAF50","trespassing":"#F44336",
          "loitering":"#FF9800","object_abandonment":"#9C27B0"}
BG, PANEL, BORDER, TEXT, MUTED = "#0D1117","#161B22","#21262D","#C9D1D9","#8B949E"

# ──────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────
print("═"*60)
print("  BIM437 · AUTO-LABEL & SAMPLE PIPELINE  v2")
print("═"*60)

cnn = r3d_18(weights=None)
cnn.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(cnn.fc.in_features, 4))
ckpt = torch.load(CNN_PATH, map_location=DEVICE)
cnn.load_state_dict(ckpt["model_state"])
cnn = cnn.to(DEVICE).eval()
print(f"✓ CNN val_f1={ckpt['val_f1']:.4f}  ·  device={DEVICE}  ·  batch={BATCH_SIZE}\n")

# ──────────────────────────────────────────────────────────
# YARDIMCILAR
# ──────────────────────────────────────────────────────────
def parse_season(day_folder_name: str):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", day_folder_name)
    if not m: return None, None
    year, month = int(m.group(1)), int(m.group(2))
    if month in SUMMER_MONTHS: return "summer", year
    if month in WINTER_MONTHS: return "winter", year
    return None, year

def extract_frames_at(cap, start_frame, n=N_FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames = []
    for _ in range(n):
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    return frames if len(frames)==n else None

def preprocess_cnn_batch(frame_lists):
    out = []
    for frames in frame_lists:
        fm  = [cv2.resize(f,(CNN_SIZE,CNN_SIZE)) for f in frames]
        arr = (np.stack(fm).astype(np.float32)/255.0 - CNN_MEAN) / CNN_STD
        out.append(arr.transpose(3,0,1,2))
    return torch.from_numpy(np.stack(out)).float()

@torch.no_grad()
def infer_cnn_batch(tensor_batch):
    tensor_batch = tensor_batch.to(DEVICE, non_blocking=True)
    with autocast('cuda'):
        logits = cnn(tensor_batch)
    return torch.sigmoid(logits).cpu().numpy()

def label_video(mp4_path, n_samples=3):
    cap = cv2.VideoCapture(str(mp4_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < N_FRAMES+1:
        cap.release(); return None, None, None
    sample_starts = np.linspace(0, max(0,total-N_FRAMES-1), n_samples, dtype=int)
    clips = []
    for s in sample_starts:
        fr = extract_frames_at(cap, int(s))
        if fr: clips.append(fr)
    cap.release()
    if not clips: return None, None, None
    tens = preprocess_cnn_batch(clips)
    probs_arr = infer_cnn_batch(tens)
    max_probs = probs_arr.max(axis=0)
    dominant = CLASSES[int(max_probs.argmax())]
    confidence = float(max_probs.max())
    return dominant, confidence, max_probs.tolist()

# ──────────────────────────────────────────────────────────
# AŞAMA 1: SHUFFLE + ERKEN DURDURMA İLE TARAMA
# ──────────────────────────────────────────────────────────
def scan_until_full():
    cache = {}
    if CACHE_FILE.exists():
        with open(CACHE_FILE) as f: cache = json.load(f)
        print(f"  📂 Cache: {len(cache)} video zaten etiketli")

    # Bucket'ları cache'ten doldur
    buckets = defaultdict(list)
    for path, info in cache.items():
        if "error" in info: continue
        if info.get("season") not in ("summer","winter"): continue
        if info.get("confidence", 0) < MIN_CONFIDENCE: continue
        buckets[(info["label"], info["season"])].append({"path":path, **info})
    for k in buckets:
        buckets[k].sort(key=lambda x:-x["confidence"])
        buckets[k] = buckets[k][:TARGET_PER_BUCKET]   # cache'ten gelse de cap

    def bucket_full(b):
        return all(len(b.get((c,s),[]))>=TARGET_PER_BUCKET
                   for c in CLASSES for s in ("summer","winter"))

    def bucket_status_line(b):
        parts = []
        for c in CLASSES:
            for s in ("summer","winter"):
                n = len(b.get((c,s),[]))
                tag = f"{c[:3]}/{s[:3]}"
                done = "✓" if n>=TARGET_PER_BUCKET else " "
                parts.append(f"{tag}:{n:>2}{done}")
        return " │ ".join(parts)

    # Gün klasörleri — shuffle
    day_dirs = [d for d in LTD_DIR.iterdir() if d.is_dir()]
    valid_days = []
    for d in day_dirs:
        season, year = parse_season(d.name)
        if season in ("summer","winter") and year in (2020,2021):
            valid_days.append((d, season, year))
    random.seed(SHUFFLE_SEED)
    random.shuffle(valid_days)
    print(f"  📁 {len(valid_days)} yaz/kış gün klasörü (shuffled)\n")

    if bucket_full(buckets):
        print("  ✓ Tüm bucket'lar zaten dolu (cache'ten)\n")
        return cache, buckets

    # Video kuyruğu
    all_videos = []
    for d, season, year in valid_days:
        mp4s = list(d.glob("*.mp4"))
        random.shuffle(mp4s)
        for mp4 in mp4s:
            all_videos.append((mp4, season, year, d.name))

    print(f"  🎞  {len(all_videos)} video kuyrukta\n")
    print(f"  Başlangıç: {bucket_status_line(buckets)}\n")

    t0 = time.time()
    processed, skipped = 0, 0

    for idx, (mp4, season, year, day_name) in enumerate(all_videos):
        key = str(mp4)

        # Bu video tipini içerecek bucket zaten dolu mu? Hızlı atla
        # (Ama label bilmeden hangi class'a düşeceğini bilmiyoruz, sadece season kontrolü yapılabilir)
        # Eğer bu mevsime ait TÜM sınıflar dolduysa videoyu işleme bile gerek yok
        season_full = all(len(buckets.get((c, season), []))>=TARGET_PER_BUCKET
                          for c in CLASSES)
        if season_full:
            continue

        if key in cache:
            skipped += 1
            # Cache'ten geliyorsa bucket'a zaten eklenmiş, devam
            continue

        try:
            label, conf, probs = label_video(mp4)
            if label is None:
                cache[key] = {"error":"no_frames"}
            else:
                cache[key] = {"label":label,"confidence":conf,"probs":probs,
                              "season":season,"year":year,
                              "day":day_name,"name":mp4.name}

                if conf >= MIN_CONFIDENCE:
                    b_key = (label, season)
                    if len(buckets[b_key]) < TARGET_PER_BUCKET:
                        buckets[b_key].append({"path":key, **cache[key]})

            processed += 1

            if processed % 20 == 0:
                with open(CACHE_FILE,"w") as f: json.dump(cache,f)
                elapsed = time.time()-t0
                rate = processed/elapsed
                remaining_buckets = sum(
                    max(0, TARGET_PER_BUCKET - len(buckets.get((c,s),[])))
                    for c in CLASSES for s in ("summer","winter")
                )
                print(f"  [{idx+1:>4}/{len(all_videos)}] proc={processed} "
                      f"({rate:.1f} v/sn) · {remaining_buckets} slot kaldı")
                print(f"  {bucket_status_line(buckets)}\n")

            if bucket_full(buckets):
                print(f"\n  🎯 TÜM BUCKET'LAR DOLU  ({idx+1}. videoda durduruldu)")
                break

        except Exception as e:
            cache[key] = {"error":str(e)[:100]}
            print(f"  ✗ {mp4.name}: {e}")

    with open(CACHE_FILE,"w") as f: json.dump(cache,f)
    print(f"\n  ✓ Tarama bitti · işlenen {processed}, atlanan {skipped}")
    print(f"  ⏱  {(time.time()-t0)/60:.1f} dakika")
    print(f"  Final: {bucket_status_line(buckets)}\n")
    return cache, buckets

# ──────────────────────────────────────────────────────────
# AŞAMA 2: BUCKET → SEÇİM DICT
# ──────────────────────────────────────────────────────────
def finalize_selection(buckets):
    selection = {}
    print("  📊 Bucket Durumu:")
    print("  " + "─"*55)
    for cls in CLASSES:
        for season in ("summer","winter"):
            vids = sorted(buckets.get((cls,season),[]),
                          key=lambda x:-x["confidence"])[:TARGET_PER_BUCKET]
            selection[f"{cls}_{season}"] = vids
            warn = "" if len(vids)>=TARGET_PER_BUCKET else f"  ⚠ ({TARGET_PER_BUCKET-len(vids)} eksik)"
            print(f"  {cls:<22} · {season:<6} : {len(vids):>3} video{warn}")
    print()
    return selection

# ──────────────────────────────────────────────────────────
# AŞAMA 3: GÖRSEL ÜRETME
# ──────────────────────────────────────────────────────────
def render_video_samples(video_info, out_dir):
    mp4 = Path(video_info["path"])
    cap = cv2.VideoCapture(str(mp4))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    sample_secs = [0, 30, 60, 90]
    sample_frames = []
    for sec in sample_secs:
        fr_idx = min(int(sec*fps), total-1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, fr_idx)
        ret,f = cap.read()
        if ret: sample_frames.append((sec, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    cap.release()
    if not sample_frames: return None

    fig, axes = plt.subplots(1,4, figsize=(16,4.5), facecolor=BG)
    for ax,(sec,frame) in zip(axes,sample_frames):
        ax.imshow(frame); ax.set_facecolor(PANEL)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"t = {sec}s", fontsize=9, color=MUTED,
                     fontfamily='monospace', pad=4)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER); sp.set_linewidth(0.6)

    title = (f"{video_info['name']}   ·   {video_info['day']}   ·   "
             f"conf={video_info['confidence']:.3f}")
    fig.suptitle(title, fontsize=10, color=TEXT,
                 fontfamily='monospace', y=0.995)
    plt.tight_layout(rect=[0,0,1,0.96])
    out_path = out_dir / f"{mp4.stem}.png"
    plt.savefig(str(out_path), dpi=110, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    return str(out_path)

def export_all_samples(selection):
    summary = {}
    t0 = time.time()
    total_videos = sum(len(v) for v in selection.values())
    done = 0
    print(f"  🖼  {total_videos} video için görsel üretiliyor...\n")

    for bucket_name, videos in selection.items():
        bucket_dir = OUTPUT_DIR / bucket_name
        bucket_dir.mkdir(exist_ok=True)
        summary[bucket_name] = []
        list_file = bucket_dir / "_video_list.txt"

        with open(list_file,"w",encoding='utf-8') as lf:
            cls, season = bucket_name.rsplit("_",1)
            lf.write(f"# {cls.upper()} · {season.upper()}\n")
            lf.write(f"# Toplam: {len(videos)} video\n")
            lf.write(f"# Oluşturma: {datetime.now().isoformat()}\n\n")
            for i,v in enumerate(videos,1):
                img_path = render_video_samples(v, bucket_dir)
                lf.write(f"{i:>3}. {v['name']}\n")
                lf.write(f"     gün: {v['day']}  ·  conf: {v['confidence']:.3f}\n")
                lf.write(f"     yol: {v['path']}\n\n")
                summary[bucket_name].append({
                    "name":v["name"], "path":v["path"], "day":v["day"],
                    "confidence":v["confidence"], "probs":v["probs"],
                    "image":img_path,
                })
                done += 1
                if done % 10 == 0:
                    rate = done/(time.time()-t0)
                    eta = (total_videos-done)/rate if rate>0 else 0
                    print(f"  [{done:>3}/{total_videos}]  {bucket_name:<32}  "
                          f"{rate:.1f} vid/sn · ETA {eta:.0f}sn")

    with open(RESULT_JSON,"w",encoding='utf-8') as f:
        json.dump(summary,f,indent=2,ensure_ascii=False)
    print(f"\n  ✓ Görseller: {OUTPUT_DIR}")
    print(f"  📄 Özet JSON: {RESULT_JSON}\n")
    return summary

def print_final_report(summary):
    print("═"*60)
    print("  FİNAL SEÇİM RAPORU")
    print("═"*60+"\n")
    for cls in CLASSES:
        for season in ("summer","winter"):
            k = f"{cls}_{season}"
            vids = summary.get(k,[])
            label = f"{cls.upper()} · {season.upper()}"
            print(f"┌─ {label}  ({len(vids)} video) " + "─"*max(0,40-len(label)))
            for i,v in enumerate(vids,1):
                print(f"│ {i:>3}. {v['name']:<30}  "
                      f"conf={v['confidence']:.3f}  ({v['day']})")
            print("└"+"─"*58+"\n")

# ──────────────────────────────────────────────────────────
# ÇALIŞTIR
# ──────────────────────────────────────────────────────────
print("[1/3] Bucket'lar dolana kadar tarama")
print("─"*60)
cache, buckets = scan_until_full()

print("[2/3] Bucket finalize")
print("─"*60)
selection = finalize_selection(buckets)

print("[3/3] Görsel + video listesi")
print("─"*60)
summary = export_all_samples(selection)

print_final_report(summary)

print("═"*60)
print(f"  TAMAM  ·  {OUTPUT_DIR}")
print("═"*60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BIM437 · TRESPASS-ONLY DETECTION
# Kural: 4x4 grid'in sol-alt hücresinde insan/sıcak nokta + hareket = trespass
# Eski trespass listesini siler, sıfırdan 40 yaz + 40 kış bulur
# ═══════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.models.video import r3d_18
from torch.amp import autocast
from pathlib import Path
from collections import defaultdict
import json, time, random, re, shutil
from datetime import datetime

# ──────────────────────────────────────────────────────────
# AYARLAR
# ──────────────────────────────────────────────────────────
LTD_DIR    = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
CNN_PATH   = "/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors-CNN/training_final/cnn_multilabel_final_model/best_model.pth"

OUTPUT_DIR  = Path("/content/drive/MyDrive/archive/auto_labeled_dataset")
CACHE_FILE  = OUTPUT_DIR / "label_cache.json"
RESULT_JSON = OUTPUT_DIR / "final_selection.json"

CLASSES        = ["normal","trespassing","loitering","object_abandonment"]
DEVICE         = torch.device("cuda")
N_FRAMES       = 16
CNN_SIZE       = 112

SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}

TARGET_PER_SEASON = 40
SHUFFLE_SEED      = 42

# Trespass karar parametreleri
CNN_PROB_MIN      = 0.30   # CNN trespass prob bu kadar olmalı (gevşek; mekansal filtre ana karar)
SPATIAL_SCORE_MIN = 0.35   # Sol-alt hücre aktivite skoru bu kadar olmalı
COMBINED_MIN      = 0.50   # CNN + spatial birleşik skor

CNN_MEAN  = np.array([0.485,0.456,0.406], dtype=np.float32)
CNN_STD   = np.array([0.229,0.224,0.225], dtype=np.float32)
BG, PANEL, BORDER, TEXT, MUTED = "#0D1117","#161B22","#21262D","#C9D1D9","#8B949E"
TRESPASS_COLOR = "#F44336"

# ──────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────
print("═"*60)
print("  TRESPASS-ONLY DETECTION  ·  sol-alt 4x4 hücre kuralı")
print("═"*60)

cnn = r3d_18(weights=None)
cnn.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(cnn.fc.in_features, 4))
ckpt = torch.load(CNN_PATH, map_location=DEVICE)
cnn.load_state_dict(ckpt["model_state"])
cnn = cnn.to(DEVICE).eval()
print(f"✓ CNN  val_f1={ckpt['val_f1']:.4f}\n")

# ──────────────────────────────────────────────────────────
# ESKİ TRESPASS VERİSİNİ TEMİZLE
# ──────────────────────────────────────────────────────────
print("🧹 Eski trespass verisi temizleniyor...")
for season in ("summer","winter"):
    old_dir = OUTPUT_DIR / f"trespassing_{season}"
    if old_dir.exists():
        shutil.rmtree(old_dir)
        print(f"   ✗ {old_dir.name} klasörü silindi")

# Cache'teki trespass etiketlerini sil → yeniden değerlendirilecek
cache = {}
if CACHE_FILE.exists():
    with open(CACHE_FILE) as f: cache = json.load(f)
    removed = 0
    for k in list(cache.keys()):
        if cache[k].get("label") == "trespassing":
            del cache[k]
            removed += 1
    with open(CACHE_FILE,"w") as f: json.dump(cache,f)
    print(f"   ✗ Cache'ten {removed} trespass etiketi silindi")

# final_selection.json güncellemesi sona bırakıldı
print()

# ──────────────────────────────────────────────────────────
# YARDIMCILAR
# ──────────────────────────────────────────────────────────
def parse_season(name):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", name)
    if not m: return None, None
    year, month = int(m.group(1)), int(m.group(2))
    if month in SUMMER_MONTHS: return "summer", year
    if month in WINTER_MONTHS: return "winter", year
    return None, year

def extract_frames(cap, start, n=N_FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(n):
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    return frames if len(frames)==n else None

def preprocess_cnn(frames):
    fm  = [cv2.resize(f,(CNN_SIZE,CNN_SIZE)) for f in frames]
    arr = (np.stack(fm).astype(np.float32)/255.0 - CNN_MEAN) / CNN_STD
    return torch.from_numpy(arr.transpose(3,0,1,2)).float()

@torch.no_grad()
def infer_cnn_batch(tensors):
    """tensors: list of (3,16,112,112) → returns (B,4) probs"""
    batch = torch.stack(tensors).to(DEVICE, non_blocking=True)
    with autocast('cuda'):
        logits = cnn(batch)
    return torch.sigmoid(logits).cpu().numpy()

# ──────────────────────────────────────────────────────────
# SOL-ALT 4x4 HÜCRE ANALİZİ
# ──────────────────────────────────────────────────────────
def bottom_left_cell(frame):
    """4x4 grid'in sol-alt hücresini döndür (x: 0-W/4, y: 3H/4-H)"""
    H, W = frame.shape[:2]
    y0, y1 = (3*H)//4, H
    x0, x1 = 0, W//4
    return frame[y0:y1, x0:x1], (x0,y0,x1,y1)

def spatial_score(frames):
    """
    Sol-alt hücrede insan/sıcak nokta + hareket sinyali.
    Termal görüntüde: parlaklık (sıcaklık) + zamanla değişim.

    Skor = 0.5 * brightness_ratio + 0.5 * motion_ratio
    """
    cells = [bottom_left_cell(f)[0] for f in frames]

    # Gri tona çevir
    gray_cells = [cv2.cvtColor(c, cv2.COLOR_RGB2GRAY) for c in cells]

    # 1) Parlaklık skoru: hücredeki ortalama vs frame ortalaması
    #    (termal'de insan vücudu daha parlak/koyu olabilir, kontrast da iyi sinyal)
    cell_means    = [g.mean() for g in gray_cells]
    full_means    = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY).mean() for f in frames]

    # Hücrenin standard sapması yüksekse içinde yapı/insan vardır
    cell_stds = [g.std() for g in gray_cells]
    avg_std = np.mean(cell_stds)
    # Sıradan boş arka plan std ~10-20, insan/araba olunca ~30-60+
    brightness_score = np.clip(avg_std / 50.0, 0, 1)

    # 2) Hareket skoru: ardışık frame'ler arası fark
    motion_vals = []
    for i in range(1, len(gray_cells)):
        diff = cv2.absdiff(gray_cells[i], gray_cells[i-1])
        # Anlamlı değişim eşiği
        motion_vals.append((diff > 15).sum() / diff.size)
    motion_score = np.clip(np.mean(motion_vals) * 5, 0, 1) if motion_vals else 0

    # 3) Maksimum parlaklık farkı (sıcak insanın varlığı)
    max_bright = max(np.percentile(g, 95) for g in gray_cells)
    bg_bright  = np.median([np.median(g) for g in gray_cells])
    contrast_score = np.clip((max_bright - bg_bright) / 80.0, 0, 1)

    combined = 0.4*motion_score + 0.3*brightness_score + 0.3*contrast_score

    return {
        "combined": float(combined),
        "motion":   float(motion_score),
        "brightness": float(brightness_score),
        "contrast": float(contrast_score),
    }

# ──────────────────────────────────────────────────────────
# VIDEO İŞLEME
# ──────────────────────────────────────────────────────────
def evaluate_video_for_trespass(mp4_path):
    """
    Bir videoyu 3 noktadan örnekle, hem CNN hem mekansal skor al.
    Skor en yüksek noktanın değerlerini döndür.
    """
    cap = cv2.VideoCapture(str(mp4_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < N_FRAMES+1:
        cap.release(); return None

    sample_starts = np.linspace(0, max(0,total-N_FRAMES-1), 3, dtype=int)

    clip_data = []  # her bir nokta için: (frames, spatial_dict)
    for s in sample_starts:
        fr = extract_frames(cap, int(s))
        if fr:
            sp = spatial_score(fr)
            clip_data.append((fr, sp))
    cap.release()

    if not clip_data: return None

    # CNN'i tüm klipler için batch'le çağır
    tensors = [preprocess_cnn(d[0]) for d in clip_data]
    probs   = infer_cnn_batch(tensors)   # (N, 4)
    trespass_idx = CLASSES.index("trespassing")

    # Her klip için combined score hesapla
    results = []
    for i, (frames, sp) in enumerate(clip_data):
        cnn_p = float(probs[i, trespass_idx])
        spatial = sp["combined"]
        # Birleşik skor: CNN ve mekansal sinyalin geometrik ortalaması
        # → ikisi de yüksekse skor yüksek olur
        combined = float(np.sqrt(cnn_p * spatial))
        results.append({
            "start": int(sample_starts[i]),
            "cnn_prob": cnn_p,
            "spatial_score": spatial,
            "spatial_detail": sp,
            "combined": combined,
            "all_probs": probs[i].tolist(),
        })

    # En yüksek combined skorlu noktayı seç
    best = max(results, key=lambda x: x["combined"])
    return best

# ──────────────────────────────────────────────────────────
# AŞAMA 1: TARAMA
# ──────────────────────────────────────────────────────────
def scan_for_trespass():
    # Tüm yaz/kış gün klasörlerini topla
    day_dirs = [d for d in LTD_DIR.iterdir() if d.is_dir()]
    valid_days = []
    for d in day_dirs:
        season, year = parse_season(d.name)
        if season in ("summer","winter") and year in (2020,2021):
            valid_days.append((d, season, year))

    random.seed(SHUFFLE_SEED)
    random.shuffle(valid_days)
    print(f"  📁 {len(valid_days)} yaz/kış gün klasörü\n")

    # Video kuyruğu
    all_videos = []
    for d, season, year in valid_days:
        mp4s = list(d.glob("*.mp4"))
        random.shuffle(mp4s)
        for mp4 in mp4s:
            all_videos.append((mp4, season, year, d.name))

    print(f"  🎞  {len(all_videos)} video kuyrukta\n")
    print(f"  Kriterler:")
    print(f"    CNN trespass prob ≥ {CNN_PROB_MIN}")
    print(f"    Sol-alt mekansal skor ≥ {SPATIAL_SCORE_MIN}")
    print(f"    Birleşik skor ≥ {COMBINED_MIN}\n")

    accepted = {"summer": [], "winter": []}
    t0 = time.time()
    processed = 0

    for idx, (mp4, season, year, day_name) in enumerate(all_videos):
        # İlgili mevsim doluysa atla
        if len(accepted[season]) >= TARGET_PER_SEASON:
            continue

        # Diğer mevsim doluysa ve sadece bu mevsim arıyoruz devam
        if all(len(accepted[s]) >= TARGET_PER_SEASON for s in ("summer","winter")):
            break

        try:
            result = evaluate_video_for_trespass(mp4)
            processed += 1

            if result is None:
                continue

            # Karar
            is_trespass = (
                result["cnn_prob"]      >= CNN_PROB_MIN and
                result["spatial_score"] >= SPATIAL_SCORE_MIN and
                result["combined"]      >= COMBINED_MIN
            )

            if is_trespass:
                entry = {
                    "path": str(mp4),
                    "name": mp4.name,
                    "day":  day_name,
                    "season": season,
                    "year": year,
                    "label": "trespassing",
                    "confidence": result["combined"],
                    "cnn_prob": result["cnn_prob"],
                    "spatial_score": result["spatial_score"],
                    "spatial_detail": result["spatial_detail"],
                    "best_start_frame": result["start"],
                    "probs": result["all_probs"],
                }
                accepted[season].append(entry)
                # Cache'e de yaz
                cache[str(mp4)] = {k:v for k,v in entry.items() if k != "path"}
                cache[str(mp4)]["probs"] = result["all_probs"]

                print(f"  ✓ [{len(accepted['summer'])}S/{len(accepted['winter'])}W] "
                      f"{mp4.name:<25} {day_name}  "
                      f"cnn={result['cnn_prob']:.2f}  spa={result['spatial_score']:.2f}  "
                      f"comb={result['combined']:.2f}")

            # Her 30 videoda checkpoint
            if processed % 30 == 0:
                with open(CACHE_FILE,"w") as f: json.dump(cache,f)
                elapsed = time.time()-t0
                rate = processed/elapsed
                print(f"\n  ── [{idx+1}/{len(all_videos)}] proc={processed} "
                      f"({rate:.1f} vid/sn) · "
                      f"sum:{len(accepted['summer'])}/{TARGET_PER_SEASON} "
                      f"win:{len(accepted['winter'])}/{TARGET_PER_SEASON}\n")

        except Exception as e:
            print(f"  ✗ {mp4.name}: {e}")

    with open(CACHE_FILE,"w") as f: json.dump(cache,f)
    print(f"\n  ⏱  {(time.time()-t0)/60:.1f} dakika · {processed} video tarandı")
    print(f"  ✓ YAZ: {len(accepted['summer'])}/{TARGET_PER_SEASON}")
    print(f"  ✓ KIŞ: {len(accepted['winter'])}/{TARGET_PER_SEASON}\n")

    return accepted

# ──────────────────────────────────────────────────────────
# AŞAMA 2: GÖRSEL ÜRETME (sol-alt hücre işaretli)
# ──────────────────────────────────────────────────────────
def render_trespass_video(video_info, out_dir):
    mp4 = Path(video_info["path"])
    cap = cv2.VideoCapture(str(mp4))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    sample_secs = [0, 30, 60, 90]
    sample_frames = []
    for sec in sample_secs:
        fr_idx = min(int(sec*fps), total-1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, fr_idx)
        ret, f = cap.read()
        if ret:
            sample_frames.append((sec, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    cap.release()
    if not sample_frames: return None

    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), facecolor=BG)
    for ax, (sec, frame) in zip(axes, sample_frames):
        ax.imshow(frame)
        ax.set_facecolor(PANEL)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"t = {sec}s", fontsize=9, color=MUTED,
                     fontfamily='monospace', pad=4)

        # Sol-alt 4x4 hücresini kırmızı çerçeveyle işaretle
        H, W = frame.shape[:2]
        rect = patches.Rectangle((0, 3*H//4), W//4, H//4,
                                  linewidth=1.8, edgecolor=TRESPASS_COLOR,
                                  facecolor='none', linestyle='--')
        ax.add_patch(rect)

        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER); sp.set_linewidth(0.6)

    title = (f"{video_info['name']}  ·  {video_info['day']}  ·  "
             f"comb={video_info['confidence']:.2f}  "
             f"(cnn={video_info['cnn_prob']:.2f} | spa={video_info['spatial_score']:.2f})")
    fig.suptitle(title, fontsize=10, color=TRESPASS_COLOR,
                 fontfamily='monospace', y=0.995)

    plt.tight_layout(rect=[0,0,1,0.96])
    out_path = out_dir / f"{mp4.stem}.png"
    plt.savefig(str(out_path), dpi=110, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    return str(out_path)

def export_trespass_samples(accepted):
    print("  🖼  Trespass görselleri üretiliyor...\n")
    t0 = time.time()
    summary = {}

    for season, videos in accepted.items():
        bucket_name = f"trespassing_{season}"
        bucket_dir = OUTPUT_DIR / bucket_name
        bucket_dir.mkdir(parents=True, exist_ok=True)
        summary[bucket_name] = []

        list_file = bucket_dir / "_video_list.txt"
        with open(list_file, "w", encoding='utf-8') as lf:
            lf.write(f"# TRESPASSING · {season.upper()}\n")
            lf.write(f"# Toplam: {len(videos)} video\n")
            lf.write(f"# Kriter: sol-alt 4x4 hücrede aktivite + CNN trespass prob\n")
            lf.write(f"# Oluşturma: {datetime.now().isoformat()}\n\n")

            # Combined skora göre sırala (en güçlüler önce)
            videos_sorted = sorted(videos, key=lambda x: -x["confidence"])

            for i, v in enumerate(videos_sorted, 1):
                img_path = render_trespass_video(v, bucket_dir)
                lf.write(f"{i:>3}. {v['name']}\n")
                lf.write(f"     gün: {v['day']}\n")
                lf.write(f"     combined={v['confidence']:.3f}  "
                         f"cnn={v['cnn_prob']:.3f}  "
                         f"spatial={v['spatial_score']:.3f}\n")
                lf.write(f"     yol: {v['path']}\n\n")
                summary[bucket_name].append({
                    **v, "image": img_path
                })

                if i % 10 == 0:
                    rate = i / (time.time()-t0) if time.time()-t0 > 0 else 0
                    print(f"  [{bucket_name}] {i}/{len(videos_sorted)}  ({rate:.1f} v/sn)")

    # final_selection.json'ı güncelle (trespass kısmını değiştir)
    full_summary = {}
    if RESULT_JSON.exists():
        with open(RESULT_JSON) as f: full_summary = json.load(f)
    full_summary["trespassing_summer"] = summary.get("trespassing_summer", [])
    full_summary["trespassing_winter"] = summary.get("trespassing_winter", [])
    with open(RESULT_JSON, "w", encoding='utf-8') as f:
        json.dump(full_summary, f, indent=2, ensure_ascii=False)

    print(f"\n  ✓ Güncellendi: {RESULT_JSON}")
    return summary

# ──────────────────────────────────────────────────────────
# RAPOR
# ──────────────────────────────────────────────────────────
def print_report(summary):
    print("\n" + "═"*60)
    print("  TRESPASS SEÇİM RAPORU")
    print("═"*60 + "\n")

    for season in ("summer","winter"):
        k = f"trespassing_{season}"
        vids = summary.get(k, [])
        print(f"┌─ TRESPASSING · {season.upper()}  ({len(vids)} video)")
        for i, v in enumerate(vids, 1):
            print(f"│ {i:>3}. {v['name']:<25}  "
                  f"comb={v['confidence']:.3f}  "
                  f"({v['day']})")
        print("└" + "─"*58 + "\n")

# ──────────────────────────────────────────────────────────
# ÇALIŞTIR
# ──────────────────────────────────────────────────────────
print("[1/2] Tüm LTD videolarını tarayıp trespass adaylarını bul")
print("─"*60)
accepted = scan_for_trespass()

print("[2/2] Görsel + liste üret (sol-alt hücre işaretli)")
print("─"*60)
summary = export_trespass_samples(accepted)

print_report(summary)

print("═"*60)
print(f"  TAMAM  ·  {OUTPUT_DIR}/trespassing_*")
print("═"*60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BIM437 · FINAL VIDEO EXPORT
# final_selection.json'dan videoları okur, sınıf klasörlerine kopyalar
# Yaz/kış ayrımı YOK · trespass klasörü BOŞ kalır
# ═══════════════════════════════════════════════════════════════════
import json, shutil, time
from pathlib import Path
from datetime import datetime

# ──────────────────────────────────────────────────────────
# AYARLAR
# ──────────────────────────────────────────────────────────
SOURCE_JSON = Path("/content/drive/MyDrive/archive/auto_labeled_dataset/final_selection.json")
OUTPUT_DIR  = Path("/content/drive/MyDrive/archive/system_video_final_test")

# Sınıf eşlemesi: json'daki anahtar prefix → klasör adı
# Yaz/kış birleştirilecek
CLASS_FOLDERS = {
    "normal":             "normal",
    "loitering":          "loitering",
    "object_abandonment": "object_abandonment",
    "trespassing":        "trespass",   # bu klasör BOŞ kalacak
}

# Trespass klasörü oluşturulur ama içine video konmaz
SKIP_CLASSES = {"trespassing"}

# ──────────────────────────────────────────────────────────
# HAZIRLIK
# ──────────────────────────────────────────────────────────
print("═"*60)
print("  FINAL VIDEO EXPORT  ·  ham mp4 kopyalama")
print("═"*60)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n  📂 Hedef: {OUTPUT_DIR}\n")

# Klasörleri oluştur (trespass dahil ama boş kalır)
for folder in CLASS_FOLDERS.values():
    (OUTPUT_DIR / folder).mkdir(exist_ok=True)
    print(f"   ✓ {folder}/")

# Json'u oku
if not SOURCE_JSON.exists():
    raise FileNotFoundError(f"Kaynak JSON yok: {SOURCE_JSON}")

with open(SOURCE_JSON, encoding='utf-8') as f:
    selection = json.load(f)

print(f"\n  📄 {SOURCE_JSON.name} okundu  ·  {len(selection)} bucket\n")

# ──────────────────────────────────────────────────────────
# KOPYALAMA
# ──────────────────────────────────────────────────────────
manifest = {
    "created_at": datetime.now().isoformat(),
    "source": str(SOURCE_JSON),
    "output_dir": str(OUTPUT_DIR),
    "videos": []  # tüm kopyalanan videoların listesi
}

stats = {cls: 0 for cls in CLASS_FOLDERS.values()}
errors = []
total_size_mb = 0

t0 = time.time()

# Her bucket'ı işle
for bucket_name, videos in selection.items():
    # bucket_name örnek: "normal_summer", "loitering_winter", "trespassing_summer"
    # son "_summer"/"_winter" kısmını at, sınıfı çıkar
    class_key = bucket_name.rsplit("_", 1)[0]  # normal, loitering, trespassing, object_abandonment

    if class_key not in CLASS_FOLDERS:
        print(f"  ⚠ Bilinmeyen sınıf: {class_key}  ({bucket_name})")
        continue

    target_folder_name = CLASS_FOLDERS[class_key]
    target_dir = OUTPUT_DIR / target_folder_name

    # Trespass: klasörü oluşturduk ama içine video koymuyoruz
    if class_key in SKIP_CLASSES:
        print(f"\n  ⏭  {bucket_name}: {len(videos)} video atlandı "
              f"(trespass klasörü boş bırakılıyor)")
        continue

    print(f"\n  📦 {bucket_name} → {target_folder_name}/")

    for v in videos:
        src = Path(v["path"])
        if not src.exists():
            errors.append(f"YOK: {src}")
            print(f"     ✗ Kaynak bulunamadı: {src.name}")
            continue

        # Hedef dosya adı: yaz/kış kaybolduğu için aynı isimde 2 video olabilir
        # Çakışma olursa numara ekle
        dst = target_dir / src.name
        if dst.exists():
            # mevsim bilgisini dosya adına gömerek çakışmayı çöz
            season = bucket_name.rsplit("_", 1)[1]   # summer/winter
            dst = target_dir / f"{src.stem}_{season}{src.suffix}"

        try:
            shutil.copy2(str(src), str(dst))
            size_mb = dst.stat().st_size / (1024*1024)
            total_size_mb += size_mb
            stats[target_folder_name] += 1

            # Manifest'e ekle
            manifest["videos"].append({
                "class": target_folder_name,
                "filename": dst.name,
                "original_path": str(src),
                "destination": str(dst),
                "size_mb": round(size_mb, 2),
                "day": v.get("day"),
                "original_season": bucket_name.rsplit("_", 1)[1],
                "confidence": v.get("confidence"),
            })

            n = stats[target_folder_name]
            print(f"     ✓ ({n:>3}) {src.name:<28}  {size_mb:>5.1f} MB")

        except Exception as e:
            errors.append(f"HATA {src.name}: {e}")
            print(f"     ✗ Kopyalama hatası: {e}")

# ──────────────────────────────────────────────────────────
# MANIFEST KAYDET (JSON + TXT)
# ──────────────────────────────────────────────────────────
manifest["stats"] = stats
manifest["total_videos"] = sum(stats.values())
manifest["total_size_mb"] = round(total_size_mb, 2)
manifest["errors"] = errors
manifest["duration_seconds"] = round(time.time() - t0, 1)

# JSON
json_path = OUTPUT_DIR / "manifest.json"
with open(json_path, "w", encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# TXT (insan-okunur)
txt_path = OUTPUT_DIR / "manifest.txt"
with open(txt_path, "w", encoding='utf-8') as f:
    f.write("═"*60 + "\n")
    f.write("  SYSTEM VIDEO FINAL TEST  ·  Manifest\n")
    f.write("═"*60 + "\n\n")
    f.write(f"Oluşturma  : {manifest['created_at']}\n")
    f.write(f"Klasör     : {OUTPUT_DIR}\n")
    f.write(f"Toplam     : {manifest['total_videos']} video · "
            f"{manifest['total_size_mb']} MB\n")
    f.write(f"Süre       : {manifest['duration_seconds']} sn\n\n")

    f.write("─"*60 + "\n")
    f.write("SINIF DAĞILIMI\n")
    f.write("─"*60 + "\n")
    for cls, n in stats.items():
        note = "  (BOŞ - manuel doldurulacak)" if cls == "trespass" else ""
        f.write(f"  {cls:<22} : {n:>3} video{note}\n")
    f.write("\n")

    # Her sınıfın detaylı listesi
    for cls in CLASS_FOLDERS.values():
        videos_in_cls = [v for v in manifest["videos"] if v["class"] == cls]
        if not videos_in_cls:
            f.write("─"*60 + "\n")
            f.write(f"{cls.upper()}  (boş)\n")
            f.write("─"*60 + "\n\n")
            continue

        f.write("─"*60 + "\n")
        f.write(f"{cls.upper()}  ({len(videos_in_cls)} video)\n")
        f.write("─"*60 + "\n")
        for i, v in enumerate(videos_in_cls, 1):
            f.write(f"{i:>3}. {v['filename']}\n")
            f.write(f"     orijinal: {v['original_path']}\n")
            f.write(f"     gün: {v['day']}  ·  mevsim: {v['original_season']}  "
                    f"·  conf: {v['confidence']:.3f}  ·  {v['size_mb']} MB\n\n")

    if errors:
        f.write("\n" + "─"*60 + "\n")
        f.write(f"HATALAR ({len(errors)})\n")
        f.write("─"*60 + "\n")
        for e in errors:
            f.write(f"  {e}\n")

# ──────────────────────────────────────────────────────────
# ÖZET
# ──────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  ÖZET")
print("═"*60)
print(f"\n  📊 Sınıf dağılımı:")
for cls, n in stats.items():
    note = "  ← BOŞ (manuel doldurulacak)" if cls == "trespass" else ""
    print(f"     {cls:<22} : {n:>3} video{note}")

print(f"\n  💾 Toplam:    {manifest['total_videos']} video  ·  "
      f"{manifest['total_size_mb']} MB")
print(f"  ⏱  Süre:      {manifest['duration_seconds']} sn")
if errors:
    print(f"  ⚠  Hata:      {len(errors)} (manifest.txt'de detay)")

print(f"\n  📂 Çıktı:     {OUTPUT_DIR}")
print(f"  📄 Manifest:  {json_path.name}  +  {txt_path.name}")
print("\n" + "═"*60)

In [ ]:
# Colab'de çalıştır:
!python /content/interactive_trespass_labeling.py

In [ ]:
import subprocess, os, json, pickle, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import cv2

LTD_DIR = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
LOCAL_FRAMES_DIR = Path("/content/frames_jpg")
LOCAL_FRAMES_DIR.mkdir(exist_ok=True)
LOCAL_FRAMES = Path("/content/all_frames.pkl")

SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}

def parse_season(day_name):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", day_name)
    if not m: return None, None, None
    year, month = int(m.group(1)), int(m.group(2))
    if month in SUMMER_MONTHS: return "summer", year, m.group(0)
    if month in WINTER_MONTHS: return "winter", year, m.group(0)
    return None, year, m.group(0)

# Video listesi
print("📁 Videoları topluyorum...")
day_dirs = [d for d in LTD_DIR.iterdir() if d.is_dir()]
all_videos = []
for d in day_dirs:
    season, year, date_str = parse_season(d.name)
    if season not in ("summer","winter") or year not in (2020,2021):
        continue
    for mp4 in d.glob("*.mp4"):
        all_videos.append({
            "path": str(mp4), "day": d.name, "season": season,
            "year": year, "name": mp4.name, "date": date_str,
            "id": mp4.stem,
        })
all_videos.sort(key=lambda x: x["date"])
print(f"   {len(all_videos)} video\n")

# Mevcut JPEG'leri kontrol
existing_jpgs = set(p.stem for p in LOCAL_FRAMES_DIR.glob("*.jpg"))
pending = [v for v in all_videos if v["id"] not in existing_jpgs]
print(f"🔄 İndirilecek: {len(pending)}")
print(f"💾 Mevcut: {len(existing_jpgs)} JPEG\n")

def extract_with_ffmpeg(video):
    """
    FFmpeg ile orta frame'i direkt al.
    -ss seek SONRA -i = fast seek (input demuxer seviyesinde)
    """
    try:
        src = video["path"]
        dst = LOCAL_FRAMES_DIR / f"{video['id']}.jpg"
        if dst.exists(): return True

        # FFmpeg ile direkt orta frame extract
        # 2dk = 120 saniye, ortası = 60sn
        cmd = [
            'ffmpeg', '-y', '-loglevel', 'error',
            '-ss', '60',                    # 60. saniyeye atla
            '-i', src,                       # Drive'dan stream
            '-frames:v', '1',                # 1 frame
            '-vf', 'scale=480:-1',           # 480px'e küçült
            '-q:v', '5',                     # JPEG kalite
            str(dst)
        ]
        result = subprocess.run(cmd, capture_output=True, timeout=30)
        return dst.exists() and dst.stat().st_size > 0
    except:
        return False

# 48 paralel ffmpeg process
WORKERS = 48
t0 = time.time()
completed = 0
success = 0

print(f"⚙️  FFmpeg ile paralel extraction ({WORKERS} worker)\n")

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = [executor.submit(extract_with_ffmpeg, v) for v in pending]
    for future in futures:
        if future.result(): success += 1
        completed += 1
        if completed % 100 == 0:
            elapsed = time.time() - t0
            rate = completed / elapsed
            eta = (len(pending) - completed) / rate if rate > 0 else 0
            print(f"   [{completed}/{len(pending)}]  {rate:.1f} fps · "
                  f"ETA {eta/60:.1f} dk · OK: {success}")

elapsed = time.time() - t0
print(f"\n✓ {success}/{len(pending)} başarılı  ·  {elapsed/60:.1f} dk")

# Pickle'a topla (etiketleme için)
print(f"\n📦 JPEG'leri pickle'a topluyorum...")
frames_db = {}
for v in all_videos:
    jpg = LOCAL_FRAMES_DIR / f"{v['id']}.jpg"
    if jpg.exists():
        with open(jpg, 'rb') as f:
            frames_db[v["path"]] = f.read()

with open(LOCAL_FRAMES, "wb") as f:
    pickle.dump(frames_db, f)

print(f"✓ {len(frames_db)} frame pickle'a yazıldı")
print(f"💾 {LOCAL_FRAMES} ({LOCAL_FRAMES.stat().st_size/(1024*1024):.0f} MB)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# AŞAMA 2: BUTON TABANLI HIZLI ETİKETLEME (ipywidgets)
# Klavye veya buton — anında cevap
# ═══════════════════════════════════════════════════════════════════
import pickle, json, shutil, cv2, re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from matplotlib.patches import Rectangle
from IPython.display import display, clear_output
import ipywidgets as widgets
import io
from PIL import Image

# ──────────────────────────────────────────────────────────
LTD_DIR = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
OUTPUT_DIR = Path("/content/drive/MyDrive/archive/system_video_final_test")
TRESPASS_DIR = OUTPUT_DIR / "trespass"
TRESPASS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_FILE = OUTPUT_DIR / "manual_trespass_labels.json"
LOCAL_LABELS = Path("/content/local_labels.json")
LOCAL_FRAMES = Path("/content/all_frames.pkl")

SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}
TARGET_PER_SEASON = 40

# ──────────────────────────────────────────────────────────
def parse_season(day_name):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", day_name)
    if not m: return None, None, None
    year, month = int(m.group(1)), int(m.group(2))
    if month in SUMMER_MONTHS: return "summer", year, m.group(0)
    if month in WINTER_MONTHS: return "winter", year, m.group(0)
    return None, year, m.group(0)

# ──────────────────────────────────────────────────────────
print("📂 Frame'ler yükleniyor...")
with open(LOCAL_FRAMES, "rb") as f:
    frames_db = pickle.load(f)
print(f"   {len(frames_db)} frame RAM'de")

# Video listesi
day_dirs = [d for d in LTD_DIR.iterdir() if d.is_dir()]
all_videos = []
for d in day_dirs:
    season, year, date_str = parse_season(d.name)
    if season not in ("summer","winter") or year not in (2020,2021):
        continue
    for mp4 in d.glob("*.mp4"):
        all_videos.append({
            "path": str(mp4), "day": d.name, "season": season,
            "year": year, "name": mp4.name, "date": date_str,
        })
all_videos.sort(key=lambda x: x["date"])

# Etiketleri yükle
labels = {}
if LOCAL_LABELS.exists():
    with open(LOCAL_LABELS) as f:
        labels = json.load(f)
elif LABEL_FILE.exists():
    with open(LABEL_FILE) as f:
        labels = json.load(f)
print(f"📂 {len(labels)} mevcut etiket")

trespass_videos = defaultdict(list)
for vpath, lbl in labels.items():
    if lbl != "trespass": continue
    for v in all_videos:
        if v["path"] == vpath:
            trespass_videos[v["season"]].append(v)
            break

def is_complete():
    return (len(trespass_videos["summer"]) >= TARGET_PER_SEASON and
            len(trespass_videos["winter"]) >= TARGET_PER_SEASON)

# Bekleyenler
pending = []
for v in all_videos:
    if v["path"] in labels: continue
    if len(trespass_videos[v["season"]]) >= TARGET_PER_SEASON: continue
    if v["path"] not in frames_db: continue
    pending.append(v)

print(f"⏳ Sırada: {len(pending)} video\n")

# ──────────────────────────────────────────────────────────
# DURUM (mutable state)
# ──────────────────────────────────────────────────────────
state = {
    "idx": 0,
    "labeled": 0,
    "done": False,
}

# ──────────────────────────────────────────────────────────
# WIDGET'LAR
# ──────────────────────────────────────────────────────────
image_widget = widgets.Image(format='jpeg', width=600)
status_label = widgets.HTML(value="")
btn_skip = widgets.Button(description='1 — NORMAL (atla)',
                           button_style='', layout=widgets.Layout(width='180px'))
btn_trespass = widgets.Button(description='2 — TRESPASS ✓',
                               button_style='danger', layout=widgets.Layout(width='180px'))
btn_quit = widgets.Button(description='5 — BİTİR',
                           button_style='warning', layout=widgets.Layout(width='180px'))
log_output = widgets.Output(layout=widgets.Layout(height='100px', overflow='auto'))

ui = widgets.VBox([
    status_label,
    image_widget,
    widgets.HBox([btn_skip, btn_trespass, btn_quit]),
    log_output,
])

# ──────────────────────────────────────────────────────────
# YARDIMCI: frame'i widget'a yükle
# ──────────────────────────────────────────────────────────
def render_frame(video):
    """Frame'i sol-alt kutuyla birlikte JPEG bytes olarak hazırla"""
    frame_bytes = frames_db[video["path"]]
    nparr = np.frombuffer(frame_bytes, np.uint8)
    bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Sol-alt 4x4 hücreyi kırmızı çerçeveyle çiz (cv2 ile, hızlı)
    H, W = bgr.shape[:2]
    cv2.rectangle(bgr, (0, 3*H//4), (W//4, H),
                  (0, 0, 255), 2)  # BGR kırmızı, kalın

    # JPEG'e encode et
    _, jpeg = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return jpeg.tobytes()

def update_status(video=None):
    s = len(trespass_videos['summer'])
    w = len(trespass_videos['winter'])
    if video:
        status_label.value = (
            f"<div style='font-family:monospace;font-size:14px'>"
            f"<b>[{state['idx']+1}/{len(pending)}]</b> "
            f"{video['date']} / {video['name']}<br>"
            f"<span style='color:#666'>{video['season'].upper()} · "
            f"oturum +{state['labeled']}</span> · "
            f"<span style='color:#FF9800'>YAZ: {s}/{TARGET_PER_SEASON}</span> · "
            f"<span style='color:#2196F3'>KIŞ: {w}/{TARGET_PER_SEASON}</span>"
            f"</div>"
        )
    else:
        status_label.value = (
            f"<div style='font-family:monospace;font-size:14px'>"
            f"<b>TAMAMLANDI</b> · YAZ: {s} · KIŞ: {w}</div>"
        )

def show_next():
    """Bir sonraki etiketlenecek videoyu göster"""
    while state["idx"] < len(pending):
        video = pending[state["idx"]]
        if video["path"] in labels:
            state["idx"] += 1
            continue
        if len(trespass_videos[video["season"]]) >= TARGET_PER_SEASON:
            state["idx"] += 1
            continue
        if is_complete():
            break
        # Bulundu
        image_widget.value = render_frame(video)
        update_status(video)
        return video

    # Bittik
    state["done"] = True
    update_status(None)
    image_widget.value = b''
    btn_skip.disabled = True
    btn_trespass.disabled = True
    with log_output:
        print("\n🎯 Hedef tamamlandı veya video bitti")
    return None

def log(msg):
    with log_output:
        print(msg)

# ──────────────────────────────────────────────────────────
# BUTON HANDLER'LARI
# ──────────────────────────────────────────────────────────
def on_skip(b=None):
    if state["done"]: return
    video = pending[state["idx"]]
    labels[video["path"]] = "not_trespass"
    state["labeled"] += 1
    state["idx"] += 1

    # Local checkpoint her 30'da
    if state["labeled"] % 30 == 0:
        with open(LOCAL_LABELS, "w") as f:
            json.dump(labels, f)

    show_next()

def on_trespass(b=None):
    if state["done"]: return
    video = pending[state["idx"]]
    labels[video["path"]] = "trespass"
    trespass_videos[video["season"]].append(video)
    state["labeled"] += 1
    state["idx"] += 1
    n = len(trespass_videos[video["season"]])
    log(f"✓ TRESPASS  {video['name']}  ({video['season']}: {n}/{TARGET_PER_SEASON})")

    if state["labeled"] % 30 == 0:
        with open(LOCAL_LABELS, "w") as f:
            json.dump(labels, f)

    show_next()

def on_quit(b=None):
    state["done"] = True
    btn_skip.disabled = True
    btn_trespass.disabled = True
    btn_quit.disabled = True
    log("\n⏹ BİTİRİLDİ — kaydediliyor...")
    save_and_copy()

# Buton bağla
btn_skip.on_click(on_skip)
btn_trespass.on_click(on_trespass)
btn_quit.on_click(on_quit)

# ──────────────────────────────────────────────────────────
# KLAVYE EVENT'İ (JS injection)
# ──────────────────────────────────────────────────────────
from IPython.display import Javascript

# Klavye dinleyici — 1, 2, 5 tuşları için
keyboard_js = """
<script>
document.addEventListener('keydown', function(e) {
    if (e.target.tagName === 'INPUT' || e.target.tagName === 'TEXTAREA') return;

    if (e.key === '1') {
        const btns = document.querySelectorAll('button');
        for (const b of btns) {
            if (b.textContent.includes('NORMAL')) { b.click(); break; }
        }
    } else if (e.key === '2') {
        const btns = document.querySelectorAll('button');
        for (const b of btns) {
            if (b.textContent.includes('TRESPASS')) { b.click(); break; }
        }
    } else if (e.key === '5') {
        const btns = document.querySelectorAll('button');
        for (const b of btns) {
            if (b.textContent.includes('BİTİR')) { b.click(); break; }
        }
    }
});
</script>
"""

# ──────────────────────────────────────────────────────────
# SONRADAN: KAYDETME + KOPYALAMA
# ──────────────────────────────────────────────────────────
def save_and_copy():
    log("📂 Etiketler Drive'a yazılıyor...")
    with open(LOCAL_LABELS, "w") as f:
        json.dump(labels, f)
    with open(LABEL_FILE, "w") as f:
        json.dump(labels, f, indent=2)
    log("✓ Etiketler kaydedildi")

    log(f"\n📦 Trespass videoları kopyalanıyor...")
    manifest = {"summer": [], "winter": []}

    for season in ("summer", "winter"):
        vids = trespass_videos[season][:TARGET_PER_SEASON]
        if not vids: continue
        log(f"   {season.upper()}: {len(vids)} video")
        for i, v in enumerate(vids, 1):
            src = Path(v["path"])
            dst = TRESPASS_DIR / src.name
            if dst.exists():
                dst = TRESPASS_DIR / f"{src.stem}_{season}{src.suffix}"
            try:
                shutil.copy2(str(src), str(dst))
                size_mb = dst.stat().st_size / (1024*1024)
                manifest[season].append({
                    "filename": dst.name, "original_path": str(src),
                    "day": v["day"], "date": v["date"],
                    "season": season, "size_mb": round(size_mb, 2),
                })
                if i % 10 == 0:
                    log(f"      {i}/{len(vids)}")
            except Exception as e:
                log(f"      ✗ {src.name}: {e}")

    with open(TRESPASS_DIR / "_trespass_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    with open(TRESPASS_DIR / "_trespass_list.txt", "w") as f:
        f.write("MANUEL TRESPASS\n" + "="*60 + "\n\n")
        for season in ("summer", "winter"):
            f.write(f"\n{season.upper()} ({len(manifest[season])})\n")
            for i, v in enumerate(manifest[season], 1):
                f.write(f"{i:>3}. {v['filename']} ({v['date']})\n")

    total = len(manifest["summer"]) + len(manifest["winter"])
    log(f"\n✓ Toplam {total} trespass kaydedildi")
    log(f"   YAZ: {len(manifest['summer'])}/{TARGET_PER_SEASON}")
    log(f"   KIŞ: {len(manifest['winter'])}/{TARGET_PER_SEASON}")
    log(f"📂 {TRESPASS_DIR}")

# ──────────────────────────────────────────────────────────
# BAŞLAT
# ──────────────────────────────────────────────────────────
print(f"\nYAZ: {len(trespass_videos['summer'])}/{TARGET_PER_SEASON}")
print(f"KIŞ: {len(trespass_videos['winter'])}/{TARGET_PER_SEASON}\n")
print("KONTROLLER:")
print("  Klavye:  1 → NORMAL  |  2 → TRESPASS  |  5 → BİTİR")
print("  Veya butonlara tıkla\n")

display(ui)
display(widgets.HTML(value=keyboard_js))

# İlk videoyu göster
show_next()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# RE-REVIEW: Belirli indeksleri tekrar göster
# 2 basılanlar mevcut trespass listesine eklenir
# ═══════════════════════════════════════════════════════════════════
import pickle, json, shutil, cv2, re
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import display
import ipywidgets as widgets

# ──────────────────────────────────────────────────────────
LTD_DIR = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
OUTPUT_DIR = Path("/content/drive/MyDrive/archive/system_video_final_test")
TRESPASS_DIR = OUTPUT_DIR / "trespass"
TRESPASS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_FILE = OUTPUT_DIR / "manual_trespass_labels.json"
LOCAL_LABELS = Path("/content/local_labels.json")
LOCAL_FRAMES = Path("/content/all_frames.pkl")

SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}

REVIEW_INDICES = [
    190, 204, 227, 662, 694, 759, 816, 846, 898, 952, 978, 999,
    1078, 1224, 1270, 1325, 1326, 1345, 1442, 1443,
    1549, 1550, 1551, 1552,
    1597, 1598, 1599, 1600, 1601,
    1611, 1631, 1662, 1671, 1673, 1674, 1694, 1699, 1723, 1741,
]

# ──────────────────────────────────────────────────────────
def parse_season(day_name):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", day_name)
    if not m: return None, None, None
    year, month = int(m.group(1)), int(m.group(2))
    if month in SUMMER_MONTHS: return "summer", year, m.group(0)
    if month in WINTER_MONTHS: return "winter", year, m.group(0)
    return None, year, m.group(0)

# Frame ve liste yükle
print("📂 Yükleniyor...")
with open(LOCAL_FRAMES, "rb") as f:
    frames_db = pickle.load(f)

day_dirs = [d for d in LTD_DIR.iterdir() if d.is_dir()]
all_videos = []
for d in day_dirs:
    season, year, date_str = parse_season(d.name)
    if season not in ("summer","winter") or year not in (2020,2021):
        continue
    for mp4 in d.glob("*.mp4"):
        all_videos.append({
            "path": str(mp4), "day": d.name, "season": season,
            "year": year, "name": mp4.name, "date": date_str,
        })
all_videos.sort(key=lambda x: x["date"])

# Pending listesini reconstruct et (önceki kodla aynı kriter)
pending_full = [v for v in all_videos if v["path"] in frames_db]
print(f"   {len(pending_full)} video listesi")

# Mevcut etiketleri yükle
labels = {}
if LOCAL_LABELS.exists():
    with open(LOCAL_LABELS) as f:
        labels = json.load(f)
elif LABEL_FILE.exists():
    with open(LABEL_FILE) as f:
        labels = json.load(f)

current_trespass = sum(1 for v in labels.values() if v == "trespass")
print(f"   Mevcut trespass: {current_trespass}\n")

# Review videolarını topla
review_videos = []
for idx in REVIEW_INDICES:
    real_idx = idx - 1
    if 0 <= real_idx < len(pending_full):
        review_videos.append((idx, pending_full[real_idx]))

print(f"🔍 {len(review_videos)} video tekrar gösterilecek\n")

# ──────────────────────────────────────────────────────────
# STATE
# ──────────────────────────────────────────────────────────
state = {
    "i": 0,
    "added": [],  # bu sessionda 2 basılanlar
    "done": False,
}

# ──────────────────────────────────────────────────────────
# WIDGETS
# ──────────────────────────────────────────────────────────
image_widget = widgets.Image(format='jpeg', width=600)
status_label = widgets.HTML(value="")
btn_skip = widgets.Button(description='1 — DEĞIL', layout=widgets.Layout(width='180px'))
btn_trespass = widgets.Button(description='2 — TRESPASS ✓',
                               button_style='danger', layout=widgets.Layout(width='180px'))
btn_quit = widgets.Button(description='5 — BİTİR',
                           button_style='warning', layout=widgets.Layout(width='180px'))
log_output = widgets.Output(layout=widgets.Layout(height='150px', overflow='auto'))

ui = widgets.VBox([
    status_label, image_widget,
    widgets.HBox([btn_skip, btn_trespass, btn_quit]),
    log_output,
])

def render_frame(video):
    frame_bytes = frames_db[video["path"]]
    nparr = np.frombuffer(frame_bytes, np.uint8)
    bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    H, W = bgr.shape[:2]
    cv2.rectangle(bgr, (0, 3*H//4), (W//4, H), (0, 0, 255), 2)
    _, jpeg = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return jpeg.tobytes()

def log(msg):
    with log_output:
        print(msg)

def show_current():
    if state["i"] >= len(review_videos):
        state["done"] = True
        status_label.value = "<b>TAMAMLANDI</b>"
        btn_skip.disabled = True
        btn_trespass.disabled = True
        log(f"\n✓ Bu sessionda {len(state['added'])} yeni trespass eklendi")
        save_and_copy()
        return

    idx, video = review_videos[state["i"]]
    image_widget.value = render_frame(video)
    status_label.value = (
        f"<div style='font-family:monospace;font-size:14px'>"
        f"<b>[{state['i']+1}/{len(review_videos)}]</b>  "
        f"orijinal indeks: <b>{idx}</b><br>"
        f"{video['date']} / {video['name']} · {video['season'].upper()}<br>"
        f"<span style='color:#F44336'>Bu sessionda eklenen: {len(state['added'])}</span>"
        f"</div>"
    )

def on_skip(b=None):
    if state["done"]: return
    state["i"] += 1
    show_current()

def on_trespass(b=None):
    if state["done"]: return
    idx, video = review_videos[state["i"]]

    # Mevcut labels'a ekle (üzerine yaz - önceki "not_trespass" idi)
    labels[video["path"]] = "trespass"
    state["added"].append(video)
    log(f"✓ #{idx} TRESPASS  {video['name']}")

    state["i"] += 1
    show_current()

def on_quit(b=None):
    state["done"] = True
    btn_skip.disabled = True
    btn_trespass.disabled = True
    log(f"\n⏹ Bitirildi — {len(state['added'])} yeni trespass")
    save_and_copy()

btn_skip.on_click(on_skip)
btn_trespass.on_click(on_trespass)
btn_quit.on_click(on_quit)

# ──────────────────────────────────────────────────────────
def save_and_copy():
    # Etiketleri kaydet (local + drive)
    with open(LOCAL_LABELS, "w") as f:
        json.dump(labels, f)
    with open(LABEL_FILE, "w") as f:
        json.dump(labels, f, indent=2)
    log("✓ Etiketler güncellendi")

    # SADECE yeni eklenen trespass videoları kopyala
    log(f"\n📦 {len(state['added'])} yeni video kopyalanıyor...")

    # Mevcut manifest'i oku (varsa)
    manifest_file = TRESPASS_DIR / "_trespass_manifest.json"
    manifest = {"summer": [], "winter": []}
    if manifest_file.exists():
        with open(manifest_file) as f:
            manifest = json.load(f)

    for v in state["added"]:
        src = Path(v["path"])
        dst = TRESPASS_DIR / src.name
        if dst.exists():
            dst = TRESPASS_DIR / f"{src.stem}_{v['season']}{src.suffix}"
        try:
            shutil.copy2(str(src), str(dst))
            size_mb = dst.stat().st_size / (1024*1024)
            manifest[v["season"]].append({
                "filename": dst.name,
                "original_path": str(src),
                "day": v["day"],
                "date": v["date"],
                "season": v["season"],
                "size_mb": round(size_mb, 2),
            })
            log(f"  ✓ {dst.name}  ({size_mb:.1f} MB)")
        except Exception as e:
            log(f"  ✗ {src.name}: {e}")

    # Manifest güncelle
    with open(manifest_file, "w", encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    # TXT güncelle
    txt_file = TRESPASS_DIR / "_trespass_list.txt"
    with open(txt_file, "w") as f:
        f.write("MANUEL TRESPASS\n" + "="*60 + "\n\n")
        for season in ("summer", "winter"):
            f.write(f"\n{season.upper()} ({len(manifest[season])})\n")
            f.write("-"*60 + "\n")
            for i, v in enumerate(manifest[season], 1):
                f.write(f"{i:>3}. {v['filename']} ({v['date']})\n")

    total = len(manifest["summer"]) + len(manifest["winter"])
    log(f"\n{'='*40}")
    log(f"TOPLAM TRESPASS: {total}")
    log(f"  YAZ: {len(manifest['summer'])}")
    log(f"  KIŞ: {len(manifest['winter'])}")
    log(f"📂 {TRESPASS_DIR}")

# Başlat
display(ui)
show_current()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# YAZ → SENTETIK KIŞ DÖNÜŞÜMÜ
# 1) Kış referans histogramı çıkar (gerçek kış trespass'lardan)
# 2) Yaz video frame'lerini kışa dönüştür (histogram + arka plan + gürültü + blur)
# 3) trespass/ ve archive/trespass_winter_synthetic/ klasörlerine kaydet
# 4) Eski yaz dosyalarını sil
# ═══════════════════════════════════════════════════════════════════
import cv2, json, shutil, random
import numpy as np
from pathlib import Path
from collections import defaultdict
import re, time

# ──────────────────────────────────────────────────────────
# AYARLAR
# ──────────────────────────────────────────────────────────
LTD_DIR = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
OUTPUT_DIR = Path("/content/drive/MyDrive/archive/system_video_final_test")
TRESPASS_DIR = OUTPUT_DIR / "trespass"
SYNTH_ARCHIVE = Path("/content/drive/MyDrive/archive/trespass_winter_synthetic")
SYNTH_ARCHIVE.mkdir(parents=True, exist_ok=True)

LABEL_FILE = OUTPUT_DIR / "manual_trespass_labels.json"
MANIFEST_FILE = TRESPASS_DIR / "_trespass_manifest.json"

N_CONVERT = 25  # yazdan kaç tanesi dönüştürülecek
SAMPLE_KEY = "20210115"  # kış referansı için sahte tarih (synthetic dosyalarda)

# ──────────────────────────────────────────────────────────
# YARDIMCILAR
# ──────────────────────────────────────────────────────────
def parse_season(day_name):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", day_name)
    if not m: return None, None
    year, month = int(m.group(1)), int(m.group(2))
    if month in {6,7,8}: return "summer", year
    if month in {12,1,2}: return "winter", year
    return None, year

# ──────────────────────────────────────────────────────────
# 1. KIŞ REFERANS HİSTOGRAMI ÇIKAR
# ──────────────────────────────────────────────────────────
def compute_winter_reference():
    """Mevcut kış trespass videolarından ortalama histogram"""
    print("📊 Kış referans histogramı hesaplanıyor...")

    # Kış trespass videolarını bul
    winter_videos = []
    for mp4 in TRESPASS_DIR.glob("*.mp4"):
        # Manifest'ten season bilgisi çek
        # veya dosya adından (eğer _winter suffix varsa)
        # En güvenlisi: manifest'i oku
        pass

    with open(MANIFEST_FILE) as f:
        manifest = json.load(f)

    winter_vids = manifest.get("winter", [])
    if not winter_vids:
        print("   ⚠ Manifestte kış trespass yok, kış normal videolardan referans alınıyor...")
        # Alternatif: LTD'den birkaç kış videosu seç
        winter_day_dirs = []
        for d in LTD_DIR.iterdir():
            if not d.is_dir(): continue
            season, year = parse_season(d.name)
            if season == "winter" and year in (2020, 2021):
                winter_day_dirs.append(d)
        random.seed(42)
        random.shuffle(winter_day_dirs)
        ref_paths = []
        for d in winter_day_dirs[:10]:
            mp4s = list(d.glob("*.mp4"))
            if mp4s:
                ref_paths.append(str(random.choice(mp4s)))
        ref_paths = ref_paths[:10]
    else:
        ref_paths = [v["original_path"] for v in winter_vids[:15]]

    print(f"   {len(ref_paths)} kış videosu referans olarak kullanılacak")

    # Her videodan birkaç frame al, histogram biriktir
    all_pixels = []
    for vpath in ref_paths:
        cap = cv2.VideoCapture(vpath)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total < 2: cap.release(); continue

        # 5 noktadan örnek
        sample_frames = np.linspace(0, total-1, 5, dtype=int)
        for fi in sample_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
            ret, frame = cap.read()
            if not ret: continue
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            # Küçült (hızlı işleme)
            gray = cv2.resize(gray, (160, 120))
            all_pixels.append(gray.flatten())
        cap.release()

    all_pixels = np.concatenate(all_pixels)

    # Kış histogramının CDF'si (histogram matching için)
    hist, _ = np.histogram(all_pixels, bins=256, range=(0, 256))
    cdf = hist.cumsum().astype(np.float64)
    cdf = cdf / cdf[-1]

    # İstatistikler
    stats = {
        "mean": float(all_pixels.mean()),
        "std": float(all_pixels.std()),
        "median": float(np.median(all_pixels)),
        "p10": float(np.percentile(all_pixels, 10)),
        "p90": float(np.percentile(all_pixels, 90)),
    }
    print(f"   Kış istatistik: mean={stats['mean']:.1f}  std={stats['std']:.1f}  "
          f"median={stats['median']:.1f}")

    return cdf, stats

# ──────────────────────────────────────────────────────────
# 2. FRAME DÖNÜŞÜM FONKSİYONU
# ──────────────────────────────────────────────────────────
def histogram_match(src_gray, ref_cdf):
    """Source frame'in pikselleri ref_cdf'e göre yeniden eşlenir"""
    src_hist, _ = np.histogram(src_gray.flatten(), bins=256, range=(0, 256))
    src_cdf = src_hist.cumsum().astype(np.float64)
    src_cdf = src_cdf / src_cdf[-1]

    # LUT: src_value → ref_value
    lut = np.zeros(256, dtype=np.uint8)
    for src_val in range(256):
        # Bu kaynak değere en yakın ref CDF değerini bul
        diff = np.abs(ref_cdf - src_cdf[src_val])
        lut[src_val] = np.argmin(diff)

    return cv2.LUT(src_gray, lut)

def yaz_to_kis(frame_bgr, ref_cdf, winter_stats):
    """
    Yaz frame'i kışa dönüştür:
    1) Gray'a çevir (termal mantığı)
    2) Histogram matching
    3) Arka plan soğutma (düşük yoğunluk = daha da koyu)
    4) Foreground koruma (sıcak nesneler)
    5) Hafif gürültü (kış sensör)
    6) Hafif blur (soft termal)
    """
    # 1. Gray
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)

    # 2. Histogram matching → kış dağılımına eşle
    matched = histogram_match(gray.astype(np.uint8), ref_cdf).astype(np.float32)

    # 3. Foreground/background ayır (Otsu)
    _, fg_mask = cv2.threshold(matched.astype(np.uint8), 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    fg_mask_f = fg_mask.astype(np.float32) / 255.0

    # Morfolojik temizlik
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    fg_mask_f = cv2.dilate(fg_mask_f, kernel, iterations=1)
    fg_mask_f = cv2.GaussianBlur(fg_mask_f, (5, 5), 0)

    # 4. Arka planı soğut (daha koyu yap), foreground'u koru
    bg_factor = 0.7   # arka plan %30 koyu
    fg_factor = 1.05  # foreground hafif parlat

    output = matched * (fg_mask_f * fg_factor + (1 - fg_mask_f) * bg_factor)
    output = np.clip(output, 0, 255)

    # 5. Kış gürültüsü (gaussian)
    noise = np.random.normal(0, 3.5, output.shape).astype(np.float32)
    output = np.clip(output + noise, 0, 255)

    # 6. Hafif blur (soft termal görüntü)
    output = cv2.GaussianBlur(output.astype(np.uint8), (3, 3), 0.5)

    # 7. Gray → BGR (3 channel)
    output_bgr = cv2.cvtColor(output, cv2.COLOR_GRAY2BGR)
    return output_bgr

# ──────────────────────────────────────────────────────────
# 3. VIDEO İŞLEME
# ──────────────────────────────────────────────────────────
def convert_video(src_path, dst_path, ref_cdf, winter_stats):
    """Tüm video frame'lerini dönüştür ve yeniden yaz"""
    cap = cv2.VideoCapture(str(src_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # mp4v codec (en uyumlu)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(dst_path), fourcc, fps, (W, H))

    if not out.isOpened():
        cap.release()
        return False, 0

    n_written = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        converted = yaz_to_kis(frame, ref_cdf, winter_stats)
        out.write(converted)
        n_written += 1

    cap.release()
    out.release()
    return True, n_written

# ──────────────────────────────────────────────────────────
# ÇALIŞTIR
# ──────────────────────────────────────────────────────────
print("═"*60)
print("  YAZ → SENTETIK KIŞ DÖNÜŞÜMÜ")
print("═"*60 + "\n")

# Kış referansı
ref_cdf, winter_stats = compute_winter_reference()

# Manifest'i oku
with open(MANIFEST_FILE) as f:
    manifest = json.load(f)

summer_vids = manifest.get("summer", [])
print(f"\n📊 Mevcut yaz trespass: {len(summer_vids)}")

if len(summer_vids) < N_CONVERT:
    print(f"⚠ {N_CONVERT} hedef, sadece {len(summer_vids)} mevcut. Hepsi dönüştürülecek.")
    N_CONVERT_REAL = len(summer_vids)
else:
    N_CONVERT_REAL = N_CONVERT

# Hangilerini dönüştüreceğiz? Random (seed'li)
random.seed(42)
to_convert = random.sample(summer_vids, N_CONVERT_REAL)
to_keep = [v for v in summer_vids if v not in to_convert]

print(f"   Dönüştürülecek: {N_CONVERT_REAL}")
print(f"   Yazda kalacak:  {len(to_keep)}\n")

# ──────────────────────────────────────────────────────────
# Dönüştür
# ──────────────────────────────────────────────────────────
print("🔄 Video dönüşümü başlıyor...\n")

converted_entries = []
errors = []
t_start = time.time()

for i, v in enumerate(to_convert, 1):
    src = TRESPASS_DIR / v["filename"]
    if not src.exists():
        # Manifest'teki orijinal path'i dene
        src = Path(v["original_path"])
        if not src.exists():
            errors.append(f"Kaynak yok: {v['filename']}")
            print(f"  ✗ [{i}/{N_CONVERT_REAL}] {v['filename']} bulunamadı")
            continue

    # Yeni dosya adı: clip_xxx_synth_winter.mp4
    stem = Path(v["filename"]).stem
    # Eğer adında _summer varsa onu temizle
    stem = stem.replace("_summer", "")
    new_name = f"{stem}_synth_winter.mp4"

    # 2 hedef
    dst_trespass = TRESPASS_DIR / new_name
    dst_archive = SYNTH_ARCHIVE / new_name

    print(f"  [{i:>2}/{N_CONVERT_REAL}] {v['filename']} → {new_name}")
    t0 = time.time()

    # Önce trespass/ klasörüne yaz
    ok, n_frames = convert_video(src, dst_trespass, ref_cdf, winter_stats)

    if not ok:
        errors.append(f"Dönüşüm başarısız: {v['filename']}")
        print(f"      ✗ Hata")
        continue

    # Archive klasörüne kopyala
    shutil.copy2(str(dst_trespass), str(dst_archive))

    elapsed = time.time() - t0
    size_mb = dst_trespass.stat().st_size / (1024*1024)
    print(f"      ✓ {n_frames} frame, {elapsed:.1f}sn, {size_mb:.1f} MB")

    # Yeni entry
    converted_entries.append({
        "filename": new_name,
        "original_path": str(src),
        "original_summer": v["filename"],
        "day": SAMPLE_KEY,  # sahte kış tarihi
        "date": SAMPLE_KEY,
        "season": "winter",
        "synthetic": True,
        "size_mb": round(size_mb, 2),
    })

    # ESKİ YAZ DOSYASINI SİL (trespass/ klasöründen)
    old_path = TRESPASS_DIR / v["filename"]
    if old_path.exists():
        old_path.unlink()
        print(f"      🗑  Eski yaz dosyası silindi: {v['filename']}")

elapsed_total = time.time() - t_start
print(f"\n⏱  Toplam dönüşüm süresi: {elapsed_total/60:.1f} dakika")

# ──────────────────────────────────────────────────────────
# Manifest'i güncelle
# ──────────────────────────────────────────────────────────
print("\n📝 Manifest güncelleniyor...")

# Yaz: dönüştürülenleri çıkar
manifest["summer"] = to_keep

# Kış: mevcutlara ekle
if "winter" not in manifest: manifest["winter"] = []
manifest["winter"].extend(converted_entries)

with open(MANIFEST_FILE, "w", encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# TXT güncelle
txt_file = TRESPASS_DIR / "_trespass_list.txt"
with open(txt_file, "w") as f:
    f.write("TRESPASS LİSTESİ (yaz + sentetik kış)\n")
    f.write("="*60 + "\n\n")
    for season in ("summer", "winter"):
        f.write(f"\n{season.upper()} ({len(manifest[season])})\n")
        f.write("-"*60 + "\n")
        for i, v in enumerate(manifest[season], 1):
            syn = " [SENTETIK]" if v.get("synthetic") else ""
            f.write(f"{i:>3}. {v['filename']}{syn}  ({v.get('date','?')})\n")

# ──────────────────────────────────────────────────────────
# ÖZET
# ──────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  ÖZET")
print("═"*60)
print(f"  Yaz trespass:        {len(manifest['summer'])} (önce {len(summer_vids)})")
print(f"  Kış trespass:        {len(manifest['winter'])}")
print(f"     gerçek:           {sum(1 for v in manifest['winter'] if not v.get('synthetic'))}")
print(f"     sentetik:         {sum(1 for v in manifest['winter'] if v.get('synthetic'))}")
print(f"  TOPLAM:              {len(manifest['summer'])+len(manifest['winter'])}")
print()
print(f"  📂 trespass:          {TRESPASS_DIR}")
print(f"  📂 sentetik arşiv:    {SYNTH_ARCHIVE}")
if errors:
    print(f"  ⚠  hatalar:           {len(errors)}")
    for e in errors[:5]:
        print(f"     · {e}")
print("═"*60)